# LightGBM 新手教程

适合：有一点机器学习基础，但还不了解 LightGBM 的同学。

目标：

1. 知道 LightGBM 是什么、适合什么问题
2. 跑通一个最小分类例子
3. 理解几个最重要的参数
4. 学会用 early stopping 和特征重要性做基本分析


## 1. LightGBM 是什么？

一句话：**LightGBM 是一种高效的 GBDT（梯度提升决策树）实现，特别适合结构化表格数据。**

常见使用场景：

- 二分类：是否流失、是否违约、是否点击
- 多分类：用户类型、商品类目
- 回归：销量预测、价格预测
- 排序：搜索排序、推荐排序

它通常不优先用于图像、语音、长文本等端到端深度学习任务。

## 2. 它和决策树、随机森林、GBDT 的关系

可以这样理解：

```text
决策树：一个 if-else 规则模型
随机森林：很多棵树并行投票，降低方差
GBDT：很多棵树串行训练，后一棵树修正前一批树的错误
LightGBM：更快、更省内存的 GBDT 实现
```

LightGBM 的关键优势：

- Histogram 分桶：连续特征先离散化，训练更快
- Leaf-wise 生长：优先分裂收益最大的叶子，通常更准
- 支持类别特征、缺失值
- 对大规模表格数据友好

## 3. 安装和导入

如果下面导入失败，在终端或 notebook 单元格里执行：

```bash
pip install lightgbm scikit-learn pandas matplotlib
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError("请先安装：pip install lightgbm") from e

print("LightGBM version:", lgb.__version__)

## 4. 准备一个分类数据集

这里用 scikit-learn 自带的乳腺癌二分类数据集。真实业务里，你通常会从 CSV、Hive、数据库或特征平台读取数据。

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(X.shape)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## 5. 训练第一个 LightGBM 模型

先用最少参数跑通流程。

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)

model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred, target_names=data.target_names))

## 6. 用 early stopping 防止训练过头

LightGBM 可以设置很多棵树，但如果验证集效果不再提升，就提前停止。

这通常比手动猜 `n_estimators` 更稳。

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

model_es = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)

model_es.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

proba_es = model_es.predict_proba(X_test)[:, 1]
print("Best iteration:", model_es.best_iteration_)
print("Test AUC:", roc_auc_score(y_test, proba_es))

## 7. 最重要的参数

新手先记这些就够了：

| 参数 | 作用 | 调大后通常怎样 |
|---|---|---|
| `n_estimators` | 树的数量 | 更容易拟合，也更慢 |
| `learning_rate` | 每棵树贡献多大 | 学得更快，但可能不稳 |
| `num_leaves` | 叶子数量 | 模型更复杂，更容易过拟合 |
| `max_depth` | 树最大深度 | 限制复杂度 |
| `min_child_samples` | 叶子最少样本数 | 更保守，不容易过拟合 |
| `subsample` | 行采样比例 | 可防过拟合 |
| `colsample_bytree` | 列采样比例 | 可防过拟合 |
| `reg_alpha` / `reg_lambda` | 正则化 | 可防过拟合 |

经验：

- 过拟合：降低 `num_leaves`，加大 `min_child_samples`，使用 `subsample` / `colsample_bytree`，加正则
- 欠拟合：增加 `num_leaves`，增加树数量，降低正则
- 常用组合：小 `learning_rate` + 大 `n_estimators` + early stopping

## 8. 查看特征重要性

特征重要性可以帮助我们粗略理解模型关注了哪些变量。

注意：特征重要性不是因果解释，只是模型使用特征的统计。

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model_es.feature_importances_,
}).sort_values("importance", ascending=False)

importance.head(10)

In [ ]:
top = importance.head(15).sort_values("importance")
plt.figure(figsize=(8, 5))
plt.barh(top["feature"], top["importance"])
plt.title("Top Feature Importance")
plt.tight_layout()
plt.show()

## 9. 一个最小调参模板

不要一开始就网格搜索几十个参数。先从下面这组开始。

In [ ]:
tuned_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=15,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)

tuned_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

proba_tuned = tuned_model.predict_proba(X_test)[:, 1]
print("Test AUC:", roc_auc_score(y_test, proba_tuned))

## 10. 新手常见坑

1. **只看 accuracy**：类别不平衡时，AUC、F1、召回率可能更重要。
2. **测试集参与调参**：测试集只能最后用一次，否则评估会虚高。
3. **`num_leaves` 太大**：LightGBM leaf-wise 生长，太复杂容易过拟合。
4. **忘记 early stopping**：树太多可能浪费时间，还可能过拟合。
5. **把特征重要性当因果**：它只能说明模型怎么用特征，不能说明业务因果。

## 11. 练习

你可以尝试：

1. 把 `num_leaves` 改成 7、15、31、63，对比 AUC。
2. 把 `learning_rate` 改成 0.1、0.05、0.01，对比最佳迭代轮数。
3. 改 `min_child_samples`，观察是否更稳。
4. 换一个 sklearn 数据集做回归任务，例如 `fetch_california_housing`。

## 12. 最短总结

LightGBM = 高效 GBDT。

学习顺序：

```text
决策树 → Boosting → GBDT → LightGBM → 调参 → 业务落地
```

新手上手时，先掌握：

- `LGBMClassifier` / `LGBMRegressor`
- `learning_rate`
- `n_estimators`
- `num_leaves`
- `min_child_samples`
- early stopping
- 训练集 / 验证集 / 测试集分开